# Poisson equation in 10D

Harmonic problem $-\Delta u = 0$ on $[0,1]^{10}$ with boundary data
$u=\sum_{i=1}^{5} x_{2i-1}x_{2i}$ (which is itself the exact solution).

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import jax
import jax.numpy as jnp

import pinn
from pinn import operators as op, sampling

%matplotlib inline
jax.config.update("jax_enable_x64", True)   # double precision

In [ ]:
class Poisson10DProblem(pinn.Problem):
    """Harmonic -Delta u = 0 on [0, 1]^10 (steady)."""

    x_min, x_max = 0.0, 1.0
    D = 10
    problem_name = "Poisson10D"
    ref_path = pinn.reference_path("poisson10d")

    def __init__(self, *, n_pde, n_bc, n_ic=0):
        self.n_pde, self.n_bc, self.n_ic = n_pde, n_bc, n_ic

    def residual_fns(self):
        return {"pde": self.pde_residual, "bc": self.bc_residual}

    def pde_residual(self, model, coords):
        u = lambda c: model(c)[0]
        return jnp.array([-op.laplacian(u, coords)])   # f = 0

    def bc_residual(self, model, coords):
        u_exact = jnp.sum(coords[0::2] * coords[1::2])
        return jnp.array([model(coords)[0] - u_exact])

    def samplers(self):
        return {"pde": sampling.box(self.x_min, self.x_max, self.D),
                "bc":  sampling.boundary_faces(self.x_min, self.x_max, self.D)}

In [ ]:
cfg = pinn.RunConfig(
    network=lambda key: pinn.SPINN(
        key, Poisson10DProblem, rank=40, hidden_dims=(20, 20, 20),
        periodic_bc=False, n_inputs=10, n_outputs=1,
    ),
    n_pde=2**16, n_bc=2**15,
    residual_sketch=5000, parameter_sketch=5000,
    batch_size=2**9, probe_batch_size=2**5,
    pde_weight=1e-4, bc_weight=1.0,
    window_scale=6.0, n_probes=64, max_radius=1e5,
)

In [ ]:
pinn.precompile64(Poisson10DProblem, cfg)

In [ ]:

results = pinn.train64(Poisson10DProblem, cfg)

## Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ref = pinn.load_reference("poisson10d")
x0, x1 = results["plot_x0"], results["plot_x1"]
lx, ly = results["plot_axes"]
extent = [x0[0], x0[-1], x1[0], x1[-1]]

for ch in ref.channels:
    pred = np.array(results["u_pred_plot"][ch])
    exact = np.array(ref.plot_grids[ch])
    err = np.abs(pred - exact)
    rel = np.linalg.norm(pred - exact) / np.linalg.norm(exact)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, data, title, cmap in zip(
        axs, [pred, exact, err],
        [f"PINN  ${ch}$", f"reference  ${ch}$", f"abs error  (rel $\\ell_2$={rel:.2e})"],
        ["RdBu_r", "RdBu_r", "magma"],
    ):
        im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap)
        ax.set(xlabel=f"${lx}$", ylabel=f"${ly}$", title=title)
        fig.colorbar(im, ax=ax)
    plt.show()